# Heaps / Priority Queues - Concepts & Use Cases

A **heap** is a specialized tree-based data structure that satisfies the heap property. A **priority queue** is an abstract data type typically implemented using a heap.

---
## Core Concepts

### Heap Property

**Min Heap**: Parent ≤ Children (smallest at root)
```
        1          ← Minimum
       / \
      3   2
     / \
    7   4
```

**Max Heap**: Parent ≥ Children (largest at root)
```
        9          ← Maximum
       / \
      7   8
     / \
    3   5
```

### Key Properties
- **Complete Binary Tree**: All levels filled except possibly last (filled left to right)
- **Array Representation**: Can be stored in array without pointers
  - Parent of `i`: `(i-1) // 2`
  - Left child of `i`: `2*i + 1`
  - Right child of `i`: `2*i + 2`

---
## Array Representation

```
Heap as tree:              Array: [1, 3, 2, 7, 4]
        1                  Index:  0  1  2  3  4
       / \
      3   2                Parent of 3 (index 1): (1-1)//2 = 0 → 1 ✓
     / \                   Left child of 1 (index 0): 2*0+1 = 1 → 3 ✓
    7   4                  Right child of 1 (index 0): 2*0+2 = 2 → 2 ✓
```

In [ ]:
import heapq
from typing import List

# Python's heapq is a MIN HEAP

# Basic operations
heap = []
heapq.heappush(heap, 3)
heapq.heappush(heap, 1)
heapq.heappush(heap, 4)
heapq.heappush(heap, 1)
heapq.heappush(heap, 5)

print(f"Heap array: {heap}")  # Note: array form, not sorted
print(f"Min element: {heap[0]}")

# Pop elements (comes out in sorted order)
sorted_list = []
while heap:
    sorted_list.append(heapq.heappop(heap))
print(f"Popped in order: {sorted_list}")

In [ ]:
# Useful heapq functions

# heapify - convert list to heap in O(n)
nums = [5, 3, 8, 1, 2, 7]
heapq.heapify(nums)
print(f"Heapified: {nums}")

# nsmallest / nlargest - get k smallest/largest
data = [10, 4, 6, 8, 2, 9, 1]
print(f"3 smallest: {heapq.nsmallest(3, data)}")
print(f"3 largest: {heapq.nlargest(3, data)}")

# heapreplace - pop and push in one operation (more efficient)
heap = [1, 3, 5, 7]
heapq.heapify(heap)
old_min = heapq.heapreplace(heap, 4)  # Pop 1, push 4
print(f"Replaced {old_min}, new heap: {heap}")

---
## Max Heap in Python

Python only provides min heap. For max heap, negate values:

In [ ]:
class MaxHeap:
    """Max heap using negation trick."""
    def __init__(self):
        self.heap = []
    
    def push(self, val):
        heapq.heappush(self.heap, -val)
    
    def pop(self):
        return -heapq.heappop(self.heap)
    
    def peek(self):
        return -self.heap[0]
    
    def __len__(self):
        return len(self.heap)

# Demo
max_heap = MaxHeap()
for val in [3, 1, 4, 1, 5, 9, 2, 6]:
    max_heap.push(val)

print("Popping from max heap:", end=" ")
while len(max_heap):
    print(max_heap.pop(), end=" ")

---
## Time & Space Complexity

| Operation | Time | Description |
|-----------|------|-------------|
| `peek` / `find-min` | O(1) | Access root |
| `push` / `insert` | O(log n) | Add + bubble up |
| `pop` / `extract-min` | O(log n) | Remove root + heapify down |
| `heapify` (build heap) | O(n) | Convert array to heap |
| `heapreplace` | O(log n) | Pop + push combined |

**Space**: O(n) to store n elements

---
## Heap Operations Visualized

### Push (Insert)
```
Insert 0 into min heap:

Step 1: Add at end       Step 2: Bubble up
        1                        0
       / \                      / \
      3   2         →          1   2
     / \                      / \
    7   0 ← new              7   3
```

### Pop (Extract Min)
```
Remove min from heap:

Step 1: Swap root/last   Step 2: Remove last   Step 3: Heapify down
        1                        7                      2
       / \                      / \                    / \
      3   2         →          3   2        →         3   7
     /                        /
    7                        (removed 1)
```

---
## Common Patterns

### 1. K Smallest/Largest Elements
- For K smallest: Use **max heap** of size K
- For K largest: Use **min heap** of size K

In [ ]:
def k_largest(nums: List[int], k: int) -> List[int]:
    """Find k largest elements using min heap of size k."""
    # Min heap keeps k largest seen so far
    # Smallest of the k largest is at top (easy to compare/replace)
    heap = nums[:k]
    heapq.heapify(heap)
    
    for num in nums[k:]:
        if num > heap[0]:  # Larger than smallest of k largest
            heapq.heapreplace(heap, num)
    
    return sorted(heap, reverse=True)  # Return in descending order

print(f"3 largest of [3,1,4,1,5,9,2,6]: {k_largest([3,1,4,1,5,9,2,6], 3)}")

### 2. Merge K Sorted Lists/Arrays
- Push first element of each list with source identifier
- Pop min, push next from same source

In [ ]:
def merge_k_sorted(lists: List[List[int]]) -> List[int]:
    """Merge k sorted lists using a min heap."""
    result = []
    # (value, list_index, element_index)
    heap = [(lst[0], i, 0) for i, lst in enumerate(lists) if lst]
    heapq.heapify(heap)
    
    while heap:
        val, list_idx, elem_idx = heapq.heappop(heap)
        result.append(val)
        
        # Push next element from same list
        if elem_idx + 1 < len(lists[list_idx]):
            next_val = lists[list_idx][elem_idx + 1]
            heapq.heappush(heap, (next_val, list_idx, elem_idx + 1))
    
    return result

lists = [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
print(f"Merged: {merge_k_sorted(lists)}")

### 3. Running Median (Two Heaps)
- Max heap for lower half
- Min heap for upper half
- Balance sizes

In [ ]:
class MedianFinder:
    def __init__(self):
        self.lo = []  # Max heap (negated) - lower half
        self.hi = []  # Min heap - upper half
    
    def add(self, num: int):
        # Add to max heap first
        heapq.heappush(self.lo, -num)
        # Move largest from lo to hi
        heapq.heappush(self.hi, -heapq.heappop(self.lo))
        # Balance: lo should have >= elements
        if len(self.hi) > len(self.lo):
            heapq.heappush(self.lo, -heapq.heappop(self.hi))
    
    def median(self) -> float:
        if len(self.lo) > len(self.hi):
            return -self.lo[0]
        return (-self.lo[0] + self.hi[0]) / 2

# Demo
mf = MedianFinder()
for num in [2, 3, 4]:
    mf.add(num)
    print(f"Added {num}, median = {mf.median()}")

---
## Real-World Use Cases

### 1. Task Scheduling / Job Queues
- **OS Process Scheduler**: Priority-based CPU scheduling
- **Job Queues**: Celery, RabbitMQ priority queues
- Higher priority tasks processed first

```
Priority Queue:
  [High] System interrupt
  [Med]  User process
  [Low]  Background task
```

### 2. Dijkstra's Shortest Path
- Always process closest unvisited node
- Min heap provides O(log V) next-node selection
- Used in GPS navigation, network routing

### 3. Huffman Coding (Data Compression)
- Build optimal prefix-free codes
- Always merge two lowest frequency nodes
- Used in ZIP, JPEG, MP3

### 4. Load Balancing
- Track server loads in heap
- Assign new request to least loaded server
- Used in web server clusters

### 5. Event-Driven Simulation
- Events ordered by timestamp
- Process earliest event first
- Used in game engines, network simulators

### 6. Median Maintenance
- Streaming data analysis
- Real-time percentile calculation
- Stock price analysis

### 7. Merge External Sorted Files
- Database merge operations
- External sorting (data > memory)
- Log aggregation systems

### 8. A* Search Algorithm
- Game pathfinding
- Priority = distance + heuristic
- Used in games, robotics

In [ ]:
# Example: Simple Task Scheduler
import time

class TaskScheduler:
    def __init__(self):
        self.tasks = []  # (priority, task_id, task_name)
        self.counter = 0
    
    def add_task(self, name: str, priority: int):
        # Lower number = higher priority
        # Counter ensures FIFO for same priority
        heapq.heappush(self.tasks, (priority, self.counter, name))
        self.counter += 1
        print(f"Added task '{name}' with priority {priority}")
    
    def run_next(self):
        if self.tasks:
            priority, _, name = heapq.heappop(self.tasks)
            print(f"Running task '{name}' (priority {priority})")
            return name
        return None

# Demo
scheduler = TaskScheduler()
scheduler.add_task("Send email", 3)
scheduler.add_task("Critical bug fix", 1)
scheduler.add_task("Update docs", 5)
scheduler.add_task("Security patch", 1)

print("\nExecuting tasks:")
while scheduler.tasks:
    scheduler.run_next()

---
## Key Tips

1. **Python heapq is MIN heap** - negate for max heap
2. **K largest → min heap of size K** (and vice versa)
3. **"Running" or "streaming" problems** often need heaps
4. **Two heaps** for median problems
5. **Heap + hashmap** for supporting decrease-key operations
6. For tuples, Python compares element by element - use counter for tiebreaking